# Phase 2: LSTM Transportation Telemetry Predictive Timing

## Project Question

**Can sequential IoT-style train telemetry improve predictive timing and delay-risk prediction compared with row-level neural-network modeling?**

This Phase 2 notebook upgrades the project with:

1. synthetic IoT-style train telemetry data  
2. Bronze/Silver/Gold data engineering pipeline  
3. sequential telemetry windows  
4. LSTM model for delay-risk classification and delay-minute regression  
5. LSTM evaluation outputs  
6. research-grade interpretation  


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))

print("Project root:", ROOT)

Project root: /Users/yuzhang/projects/Machine_learning/01_transportation_telemetry_predictive_timing_phase2


## 1. Generate IoT-style train telemetry data

In [2]:
from src.data_generator import generate_synthetic_telemetry

df = generate_synthetic_telemetry(
    n_events=12000,
    output_path=ROOT / "data/raw/train_telemetry_events.csv",
)

display(df.head())
print(df.shape)

,event_id,event_time,train_id,route_id,latitude,longitude,scheduled_minutes,distance_miles,avg_speed_mph,brake_pressure,engine_temp,vibration_score,weather_severity,route_congestion,cargo_weight_tons,delay_minutes,delay_risk
0,EVT_00000000,2026-01-03 16:15:00,TRAIN_0047,ROUTE_014,32.043939,-81.091223,133.335274,260.480373,85.000000,100.939544,166.125967,3.261658,0,0.432211,6527.121705,14.417065,0
1,EVT_00000001,2026-01-24 05:14:00,TRAIN_0188,ROUTE_008,36.989359,-101.630206,201.718597,368.791409,85.000000,86.814082,137.727534,0.959339,1,0.363819,6386.465794,25.569142,1
2,EVT_00000002,2026-01-20 15:17:00,TRAIN_0018,ROUTE_023,39.329757,-95.250839,152.661193,113.409843,44.897004,82.913474,155.378223,1.458856,1,0.184398,8440.695578,21.104442,1
3,EVT_00000003,2026-01-14 03:59:00,TRAIN_0126,ROUTE_012,34.832034,-91.420129,105.792015,278.441715,85.000000,90.115093,194.960310,0.578870,0,0.563730,7202.354009,15.375159,0
4,EVT_00000004,2026-01-13 23:46:00,TRAIN_0123,ROUTE_014,39.528934,-101.394562,167.063799,297.346892,85.000000,89.471863,164.933348,0.385409,2,0.255807,8481.008762,5.723146,0


(12000, 17)


## 2. Run Bronze/Silver/Gold lakehouse-style pipeline

In [3]:
from src.pipeline import bronze_ingest, silver_clean, gold_features

bronze_df = bronze_ingest(
    ROOT / "data/raw/train_telemetry_events.csv",
    ROOT / "data/bronze/telemetry_bronze.parquet"
)

silver_df = silver_clean(
    ROOT / "data/bronze/telemetry_bronze.parquet",
    ROOT / "data/silver/telemetry_silver.parquet"
)

gold_df = gold_features(
    ROOT / "data/silver/telemetry_silver.parquet",
    ROOT / "data/gold/train_delay_features.parquet"
)

display(gold_df.head())
print(gold_df.shape)

,event_id,event_time,train_id,route_id,latitude,longitude,scheduled_minutes,distance_miles,avg_speed_mph,brake_pressure,...,weather_severity,route_congestion,cargo_weight_tons,delay_minutes,delay_risk,hour,day_of_week,route_avg_delay,route_avg_congestion,route_event_count
0,EVT_00000000,2026-01-03 16:15:00,TRAIN_0047,ROUTE_014,32.043939,-81.091223,133.335274,260.480373,85.000000,100.939544,...,0,0.432211,6527.121705,14.417065,0,16,5,16.215306,0.276686,426
1,EVT_00000001,2026-01-24 05:14:00,TRAIN_0188,ROUTE_008,36.989359,-101.630206,201.718597,368.791409,85.000000,86.814082,...,1,0.363819,6386.465794,25.569142,1,5,5,16.317119,0.291464,429
2,EVT_00000002,2026-01-20 15:17:00,TRAIN_0018,ROUTE_023,39.329757,-95.250839,152.661193,113.409843,44.897004,82.913474,...,1,0.184398,8440.695578,21.104442,1,15,1,16.279845,0.280118,369
3,EVT_00000003,2026-01-14 03:59:00,TRAIN_0126,ROUTE_012,34.832034,-91.420129,105.792015,278.441715,85.000000,90.115093,...,0,0.563730,7202.354009,15.375159,0,3,2,16.080495,0.305407,423
4,EVT_00000004,2026-01-13 23:46:00,TRAIN_0123,ROUTE_014,39.528934,-101.394562,167.063799,297.346892,85.000000,89.471863,...,2,0.255807,8481.008762,5.723146,0,23,1,16.215306,0.276686,426


(12000, 22)


## 3. Create sequential telemetry windows

In [ ]:
from src.sequence_features import create_lstm_sequences

X_seq, y_risk, y_delay, metadata = create_lstm_sequences(
    gold_path=ROOT / "data/gold/train_delay_features.parquet",
    sequence_path=ROOT / "data/gold/train_delay_sequences.npz",
    scaler_path=ROOT / "outputs/models/lstm_sequence_scaler.joblib",
    metadata_path=ROOT / "outputs/tables/lstm_sequence_metadata.json",
    sequence_length=8,
)

print("X_seq shape:", X_seq.shape)
print("Risk labels:", y_risk.shape)
print("Delay labels:", y_delay.shape)
metadata

## 4. Train LSTM predictive timing model

In [ ]:
from src.lstm_model import train_lstm_model

lstm_metrics = train_lstm_model(
    sequence_path=ROOT / "data/gold/train_delay_sequences.npz",
    model_path=ROOT / "outputs/models/lstm_delay_timing.pt",
    metrics_path=ROOT / "outputs/tables/lstm_model_metrics.json",
    report_path=ROOT / "outputs/tables/lstm_classification_report.csv",
    predictions_path=ROOT / "outputs/tables/lstm_predictions.csv",
    loss_path=ROOT / "outputs/tables/lstm_training_loss.csv",
    confusion_matrix_path=ROOT / "outputs/tables/lstm_confusion_matrix.csv",
    epochs=40,
)

lstm_metrics

## 5. Generate LSTM visual outputs

In [ ]:
from src.lstm_visualization import generate_lstm_figures

figures = generate_lstm_figures(
    predictions_path=ROOT / "outputs/tables/lstm_predictions.csv",
    loss_path=ROOT / "outputs/tables/lstm_training_loss.csv",
    metrics_path=ROOT / "outputs/tables/lstm_model_metrics.json",
    output_dir=ROOT / "outputs/figures",
)

figures

## 6. Review outputs

In [ ]:
import pandas as pd

display(pd.read_json(ROOT / "outputs/tables/lstm_model_metrics.json", typ="series"))
display(pd.read_csv(ROOT / "outputs/tables/lstm_classification_report.csv"))
display(pd.read_csv(ROOT / "outputs/tables/lstm_predictions.csv").head())

## Final Interpretation

This Phase 2 upgrade makes the project more research-aligned because the LSTM model uses **sequential telemetry history** instead of one event row at a time.

This better reflects railway and transportation predictive maintenance settings where delay risk, equipment degradation, and operational disruptions emerge from temporal patterns such as:

- rolling train movement state
- speed history
- vibration trends
- temperature history
- congestion progression
- weather exposure over time

This remains a synthetic portfolio project, but the model architecture is now closer to railway predictive maintenance and sequence forecasting research.
